In [ ]:
#  Write a program to demonstrate the working of the decision tree based ID3 algorithm. Use an appropriate data set for building the decision tree and apply this knowledge to classify a new sample.

In [1]:
import pandas as pd
import numpy as np
import pprint

eps = np.finfo(float).eps
from numpy import log2 as log
def find_entropy(df):
    Class = df.keys()[-1]
    entropy = 0
    values = df[Class].unique()
    
    for value in values:
        fraction = df[Class].value_counts()[value] / len(df[Class])
        entropy += -fraction * np.log2(fraction)
        
    return entropy
def find_entropy_attribute(df, attribute):
    Class = df.keys()[-1]
    target_variables = df[Class].unique()
    variables = df[attribute].unique()
    entropy2 = 0
    
    for variable in variables:
        entropy = 0
        
        for target_variable in target_variables:
            num = len(df[(df[attribute] == variable) & (df[Class] == target_variable)])
            den = len(df[df[attribute] == variable])
            fraction = num / (den + eps)
            entropy += -fraction * log(fraction + eps)
            
        fraction2 = den / len(df)
        entropy2 += fraction2 * entropy
        
    return abs(entropy2)
def find_winner(df):
    IG = []
    
    for key in df.keys()[:-1]:
        ig = find_entropy(df) - find_entropy_attribute(df, key)
        IG.append(ig)
        
    return df.keys()[:-1][np.argmax(IG)]
def get_subtable(df, node, value):
    return df[df[node] == value].reset_index(drop=True)

def buildTree(df, tree=None):
    Class = df.keys()[-1]
    
    node = find_winner(df)
    attValue = np.unique(df[node])
    
    if tree is None:
        tree = {}
        tree[node] = {}
        
    for value in attValue:
        subtable = get_subtable(df, node, value)
        clValue, counts = np.unique(subtable[Class], return_counts=True)
        
        if len(counts) == 1:
            tree[node][value] = clValue[0]
        else:
            tree[node][value] = buildTree(subtable)
            
    return tree


df = pd.read_csv('playtennis.csv')

print("\nGiven Play Tennis Data Set:\n")
print(df.to_string())


tree = buildTree(df)

print("\nResultant Decision Tree:\n")
pprint.pprint(tree)

test = {
    'Outlook': 'Sunny',
    'Temperature': 'Hot',
    'Humidity': 'High',
    'Wind': 'Weak'
}

def classify(test, tree, default=None):
    attribute = next(iter(tree))
    
    print(f"\nChecking Attribute: {attribute}")
    print(f"Possible Values: {list(tree[attribute].keys())}")
    print(f"Test Value: {test[attribute]}")
    
    if test[attribute] in tree[attribute]:
        result = tree[attribute][test[attribute]]
        
        if isinstance(result, dict):
            return classify(test, result)
        else:
            print(f"\nReached Leaf Node → Decision: {result}")
            return result
    else:
        return default
ans = classify(test, tree)
print("\nFinal Prediction:", ans)



Given Play Tennis Data Set:

     Outlook Temperature Humidity    Wind PlayTennis
0      Sunny         Hot     High    Weak         No
1      Sunny         Hot     High  Strong         No
2   Overcast         Hot     High    Weak        Yes
3       Rain        Mild     High    Weak        Yes
4       Rain        Cool   Normal    Weak        Yes
5       Rain        Cool   Normal  Strong         No
6   Overcast        Cool   Normal  Strong        Yes
7      Sunny        Mild     High    Weak         No
8      Sunny        Cool   Normal    Weak        Yes
9       Rain        Mild   Normal    Weak        Yes
10     Sunny        Mild   Normal  Strong        Yes
11  Overcast        Mild     High  Strong        Yes
12  Overcast         Hot   Normal    Weak        Yes
13      Rain        Mild     High  Strong         No

Resultant Decision Tree:

{'Outlook': {'Overcast': 'Yes',
             'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}},
             'Sunny': {'Humidity': {'High': 'No', 'N